# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Notebook này đã triển khai các bước trong `GUIDE.md`. Sau khi chọn GPU T4, cấu hình Kaggle Secrets và bấm **Run all**, notebook chạy E0–E4, lưu evidence/biểu đồ vào `report/`, sao lưu checkpoint và kết quả lên Google Drive, rồi nghiệm thu bằng `scripts/check_submission.py`. E5 là bonus: đã có reward và unit checks, nhưng chỉ train khi bật cờ riêng.
Luồng đầy đủ có thể kéo dài vài giờ; hãy giữ runtime và Drive mount hoạt động. Notebook tự điền bảng số liệu trong `REPORT.md` và tạo `report/day10_report_handoff.zip` để tải về, giải nén tại repo root; 10 câu phân tích và họ tên vẫn cần bạn hoàn thiện bằng kết quả thật.


## 0. Môi trường (Colab T4)

In [ ]:
# 0.1 Mount Google Drive để lưu checkpoint/adapter đề phòng mất session
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 0.2 Đồng bộ mã nguồn repo nếu chạy trên Colab
import os
REPO_DIR = "/content/K4-Track-4-Day10-VisionLanguageModel"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/awun0105/K4-Track-4-Day10-VisionLanguageModel.git /content/K4-Track-4-Day10-VisionLanguageModel
%cd /content/K4-Track-4-Day10-VisionLanguageModel
# Nếu đã clone từ trước, fast-forward để lấy notebook mới; thay đổi local sẽ làm pull dừng an toàn.
!git pull --ff-only origin main

In [ ]:
# 0.3 Kiểm tra T4 GPU và cài đặt thư viện
# Lưu ý: Chọn menu Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!python -m pytest -q


In [ ]:
# 0.4 Cấu hình thư mục lưu checkpoint trên Google Drive
import os
import shutil
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/AI20K_Lab_Day10_Checkpoints"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

def backup_to_drive(src_path, exp_name):
    """Hàm tiện ích sao lưu checkpoint/adapter sang Google Drive"""
    dest = os.path.join(DRIVE_BACKUP_DIR, exp_name)
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest, dirs_exist_ok=True)
    elif os.path.isfile(src_path):
        shutil.copy2(src_path, dest)
    print(f"✅ Đã sao lưu {src_path} -> {dest}")

def restore_from_drive(exp_name, dest_path):
    """Khôi phục artifact đã sao lưu khi Colab tạo runtime mới."""
    src_path = os.path.join(DRIVE_BACKUP_DIR, exp_name)
    if not os.path.exists(src_path):
        return False
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest_path, dirs_exist_ok=True)
    else:
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        shutil.copy2(src_path, dest_path)
    print(f"Đã khôi phục {exp_name} từ Google Drive.")
    return True


In [ ]:
# 0.5 Kaggle & Hugging Face credentials: nạp từ Colab Secrets hoặc file .env / biến môi trường
# TUÂN THỦ QUY ĐỊNH: Tuyệt đối KHÔNG ghi cứng token / secret key vào code.
import os

# 1. Nạp từ Colab Secrets (khi chạy trên Google Colab)
try:
    from google.colab import userdata
    for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"]:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    pass

# 2. Nạp từ file .env (khi chạy local / Google Drive mount có .env)
env_file = ".env"
if os.path.exists(env_file):
    try:
        with open(env_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    k, v = k.strip(), v.strip().split("#")[0].strip()
                    if k and v and k not in os.environ:
                        os.environ[k] = v
    except Exception:
        pass

# Đồng bộ token cho Hugging Face Hub
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    try:
        from huggingface_hub import login
        login(token=os.environ["HF_TOKEN"])
        print("✅ Đã xác thực Hugging Face Hub thành công.")
    except Exception as e:
        print(f"Lưu ý Hugging Face login: {e}")

print("Kaggle Username:", "Đã thiết lập" if os.environ.get("KAGGLE_USERNAME") else "Chưa có (cần đặt trong Colab Secrets)")
print("Kaggle Key:", "Đã thiết lập" if os.environ.get("KAGGLE_KEY") else "Chưa có (cần đặt trong Colab Secrets)")
print("HF Token:", "Đã thiết lập" if os.environ.get("HF_TOKEN") else "Chưa có (tuỳ chọn)")


## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# GUIDE bước 1: kiểm tra split theo session, số ảnh/box và vẽ 6 ảnh có nhãn.
import json
import random
from collections import Counter
from pathlib import Path

from PIL import Image, ImageDraw
import matplotlib.pyplot as plt


def audit_data_splits(data_dir: str = "data", example_count: int = 6, seed: int = 42):
    data_root = Path(data_dir)
    split_rows = {}
    for split in ("train", "val", "test"):
        split_path = data_root / "splits" / f"{split}.jsonl"
        if not split_path.is_file():
            raise FileNotFoundError(f"Thiếu split: {split_path}; chạy scripts/prepare_data.py trước.")
        with split_path.open(encoding="utf-8") as source:
            split_rows[split] = [json.loads(line) for line in source if line.strip()]
        if not split_rows[split]:
            raise ValueError(f"Split {split} rỗng: {split_path}")

    missing_sessions = [
        (split, row.get("image_id"))
        for split, rows in split_rows.items()
        for row in rows
        if not row.get("session")
    ]
    if missing_sessions:
        raise ValueError(f"Không thể xác nhận split theo session; thiếu session ở {missing_sessions[:5]}")

    sessions_by_split = {
        split: {row.get("session") for row in rows if row.get("session") is not None}
        for split, rows in split_rows.items()
    }
    overlaps = {
        f"{left}/{right}": sorted(sessions_by_split[left] & sessions_by_split[right])
        for left, right in (("train", "val"), ("train", "test"), ("val", "test"))
        if sessions_by_split[left] & sessions_by_split[right]
    }
    if overlaps:
        raise ValueError(f"Có session xuất hiện ở nhiều split: {overlaps}")

    summary = {}
    missing_images = []
    size_mismatches = []
    for split, rows in split_rows.items():
        classes = Counter(obj["label"] for row in rows for obj in row.get("objects", []))
        summary[split] = {
            "images": len(rows),
            "boxes": sum(len(row.get("objects", [])) for row in rows),
            "boxes_by_class": dict(sorted(classes.items())),
            "sessions": len(sessions_by_split[split]),
        }
        for row in rows:
            image_path = data_root / row["image_path"]
            if not image_path.is_file():
                missing_images.append(str(image_path))
                continue
            with Image.open(image_path) as image:
                if (image.width, image.height) != (int(row["width"]), int(row["height"])):
                    size_mismatches.append((row["image_id"], (image.width, image.height),
                                            (row["width"], row["height"])))
    if missing_images:
        raise FileNotFoundError(f"Thiếu {len(missing_images)} ảnh, ví dụ: {missing_images[:3]}")
    if size_mismatches:
        raise ValueError(f"Kích thước ảnh không khớp metadata, ví dụ: {size_mismatches[:3]}")

    meta_path = data_root / "splits" / "meta.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.is_file() else {}
    data_audit_artifact = {
        "dataset_version": meta.get("dataset_version"),
        "session_disjoint": True,
        "splits": summary,
    }
    report_dir = Path("report")
    figures_dir = report_dir / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)
    audit_path = report_dir / "data_audit.json"
    audit_path.write_text(
        json.dumps(data_audit_artifact, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    if "backup_to_drive" in globals():
        backup_to_drive(str(audit_path), "data_audit.json")
    print(f"dataset_version={meta.get('dataset_version', 'không có meta.json')}")
    for split, stats in summary.items():
        print(f"{split}: {stats['images']} ảnh, {stats['boxes']} box, "
              f"{stats['sessions']} sessions, lớp={stats['boxes_by_class']}")
    print("PASS: session không chồng lấn; đường dẫn và kích thước ảnh khớp metadata.")

    train_rows = split_rows["train"]
    selected = random.Random(seed).sample(train_rows, k=min(example_count, len(train_rows)))
    columns = min(3, len(selected))
    rows_count = (len(selected) + columns - 1) // columns
    figure, axes = plt.subplots(rows_count, columns, figsize=(6 * columns, 4.5 * rows_count), squeeze=False)
    for axis, row in zip(axes.flat, selected):
        image_path = data_root / row["image_path"]
        with Image.open(image_path) as source_image:
            image = source_image.convert("RGB")
        drawing = ImageDraw.Draw(image)
        for obj in row.get("objects", []):
            xc, yc, width, height = (float(value) for value in obj["yolo"])
            box = (
                round((xc - width / 2) * image.width),
                round((yc - height / 2) * image.height),
                round((xc + width / 2) * image.width),
                round((yc + height / 2) * image.height),
            )
            drawing.rectangle(box, outline="red", width=3)
            drawing.text((box[0], max(0, box[1] - 14)), obj["label"], fill="red")
        axis.imshow(image)
        axis.set_title(f"{row['image_id']} | {len(row.get('objects', []))} box")
        axis.axis("off")
    for axis in axes.flat[len(selected):]:
        axis.axis("off")
    figure.tight_layout()
    gt_figure_path = figures_dir / "train_gt_examples.png"
    figure.savefig(gt_figure_path, dpi=140)
    if "backup_to_drive" in globals():
        backup_to_drive(str(gt_figure_path), "figure_train_gt_examples.png")
    plt.show()
    plt.close(figure)
    print(f"Đã lưu split audit: report/data_audit.json; ảnh GT: {figures_dir / 'train_gt_examples.png'}")
    return data_audit_artifact


data_audit = audit_data_splits()


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json
import math


def _scale_yolo_box(yolo: list[float], width: int, height: int) -> list[int]:
    if len(yolo) != 4:
        raise ValueError(f"YOLO box phải có 4 giá trị, nhận được {len(yolo)}")
    xc, yc, bw, bh = (float(value) for value in yolo)
    if not all(math.isfinite(value) for value in (xc, yc, bw, bh)):
        raise ValueError(f"YOLO box có giá trị không hữu hạn: {yolo}")
    if bw <= 0 or bh <= 0:
        raise ValueError(f"YOLO box phải có width/height dương: {yolo}")

    coords = [
        round((xc - bw / 2.0) * width),
        round((yc - bh / 2.0) * height),
        round((xc + bw / 2.0) * width),
        round((yc + bh / 2.0) * height),
    ]
    x1, y1, x2, y2 = [max(0, min(limit, value)) for value, limit in zip(coords, (width, height, width, height))]
    if x2 <= x1 or y2 <= y1:
        raise ValueError(f"Box suy biến sau khi kẹp vào ảnh: {yolo} -> {[x1, y1, x2, y2]}")
    return [x1, y1, x2, y2]


def yolo_to_box(label: str, yolo: list[float]) -> dict:
    """Chuyển YOLO [xc, yc, w, h] sang toạ độ grounding nguyên trên thang 0–1000."""
    return {"label": label, "bbox_2d": _scale_yolo_box(yolo, 1000, 1000)}


def yolo_to_pixel_box(label: str, yolo: list[float], width: int = 640, height: int = 360) -> dict:
    """Chuyển YOLO sang pixel [x1, y1, x2, y2] của ảnh có kích thước width × height."""
    return {"label": label, "bbox_2d": _scale_yolo_box(yolo, width, height)}


def objects_to_answer(
    objects: list[dict],
    coord_scale: str = "norm1000",
    width: int = 640,
    height: int = 360,
) -> str:
    """Đổi nhãn thành JSON ổn định theo thứ tự trên–dưới, trái–phải."""
    if coord_scale == "norm1000":
        boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    elif coord_scale == "pixel":
        boxes = [yolo_to_pixel_box(obj["label"], obj["yolo"], width, height) for obj in objects]
    else:
        raise ValueError(f"coord_scale không hợp lệ: {coord_scale}")

    boxes.sort(key=lambda box: (box["bbox_2d"][1], box["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"), ensure_ascii=False)

def verify_coordinate_roundtrip(
    split_path: str = "data/splits/train.jsonl",
    output_path: str = "report/figures/coord_roundtrip.png",
):
    """Vẽ pixel gốc và YOLO→0–1000→pixel; sai lệch cho phép tối đa 1 px do làm tròn."""
    from PIL import Image, ImageDraw
    from pathlib import Path

    with open(split_path, encoding="utf-8") as source:
        rows = [json.loads(line) for line in source if line.strip()]
    chosen = next((row for row in rows if row.get("objects")), None)
    if chosen is None:
        raise ValueError("Không có box để kiểm tra round-trip tọa độ.")
    obj = chosen["objects"][0]
    direct = _scale_yolo_box(obj["yolo"], int(chosen["width"]), int(chosen["height"]))
    norm_box = yolo_to_box(obj["label"], obj["yolo"])["bbox_2d"]
    width, height = int(chosen["width"]), int(chosen["height"])
    roundtrip = [
        round(norm_box[0] * width / 1000), round(norm_box[1] * height / 1000),
        round(norm_box[2] * width / 1000), round(norm_box[3] * height / 1000),
    ]
    max_error_px = max(abs(a - b) for a, b in zip(direct, roundtrip, strict=True))
    if max_error_px > 1:
        raise AssertionError(f"Sai lệch round-trip {max_error_px}px vượt quá 1px: {direct} vs {roundtrip}")

    with Image.open(Path("data") / chosen["image_path"]) as source_image:
        image = source_image.convert("RGB")
    draw = ImageDraw.Draw(image)
    draw.rectangle(direct, outline="lime", width=4)
    draw.text((direct[0], max(0, direct[1] - 16)), f"YOLO pixel {obj['label']}", fill="lime")
    draw.rectangle(roundtrip, outline="magenta", width=2)
    draw.text((roundtrip[0], min(height - 14, roundtrip[1] + 2)), "round-trip", fill="magenta")
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    image.save(output)
    result = {
        "image_id": chosen["image_id"], "label": obj["label"],
        "direct_pixel_box": direct, "roundtrip_pixel_box": roundtrip,
        "max_error_px": max_error_px, "pass": True, "figure": str(output),
    }
    Path("report/coordinate_roundtrip.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"PASS coordinate round-trip: {direct} -> {norm_box} -> {roundtrip}; lệch tối đa {max_error_px}px.")
    print(f"Đã lưu hình kiểm tra: {output}")
    if "backup_to_drive" in globals():
        backup_to_drive(str(output), "figure_coord_roundtrip.png")
        backup_to_drive("report/coordinate_roundtrip.json", "coordinate_roundtrip.json")
    return result


coordinate_roundtrip = verify_coordinate_roundtrip()


In [ ]:
SYSTEM_PROMPT = (
    "You are a road inspection assistant. Detect all road damage objects in the image. "
    "Return only a JSON list. Each item must have 'label' and 'bbox_2d'. "
    "Allowed labels: 'pothole', 'crack', 'manhole'. "
    "Coordinates must be integers on the 0–1000 scale in [x1, y1, x2, y2] order."
)

PIXEL_SYSTEM_PROMPT = (
    "You are a road inspection assistant. Detect all road damage objects in the image. "
    "Return only a JSON list. Each item must have 'label' and 'bbox_2d'. "
    "Allowed labels: 'pothole', 'crack', 'manhole'. "
    "Coordinates must be integer pixel values in [x1, y1, x2, y2] order, "
    "with x and y bounded by the actual image width and height."
)


def system_prompt_for_scale(coord_scale: str) -> str:
    if coord_scale == "norm1000":
        return SYSTEM_PROMPT
    if coord_scale == "pixel":
        return PIXEL_SYSTEM_PROMPT
    raise ValueError(f"coord_scale không hợp lệ: {coord_scale}")

## 3. E0: zero-shot

In [ ]:
# Nạp model 4-bit cho E0 và dùng lại cùng cấu hình khi huấn luyện/đánh giá E1–E3.
import json
import math
from pathlib import Path

import torch
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

model_id = "Qwen/Qwen3-VL-4B-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    # Theo GUIDE: giữ visual và lm_head không bị lượng tử hoá.
    llm_int8_skip_modules=["visual", "lm_head"],
)


def load_base_model():
    if not torch.cuda.is_available():
        raise RuntimeError("Lab yêu cầu GPU CUDA; hãy chọn runtime T4 trên Colab.")
    return Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": 0},
    )


def _chunks(items, batch_size: int):
    if batch_size < 1:
        raise ValueError("batch_size phải lớn hơn 0")
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def write_predictions_jsonl(
    infer_model,
    entries: list[dict],
    output_path: str | Path,
    prompt_text: str,
    batch_size: int = 1,
    max_new_tokens: int = 256,
    desc: str = "Suy luận",
) -> Path:
    """Sinh JSONL theo batch; padding trái giúp lấy đúng phần token model mới sinh."""
    if batch_size < 1:
        raise ValueError("batch_size phải lớn hơn 0")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    batches = math.ceil(len(entries) / batch_size)

    with output_path.open("w", encoding="utf-8") as out_file:
        for batch_entries in tqdm(_chunks(entries, batch_size), total=batches, desc=desc):
            images = []
            for item in batch_entries:
                with Image.open(Path("data") / item["image_path"]) as source_image:
                    images.append(source_image.convert("RGB"))

            messages = [
                [{"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text},
                ]}]
                for image in images
            ]
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to("cuda")
            prompt_width = inputs["input_ids"].shape[1]

            with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
                generated = infer_model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                )
            raw_outputs = processor.batch_decode(
                generated[:, prompt_width:],
                skip_special_tokens=True,
            )
            for item, raw in zip(batch_entries, raw_outputs, strict=True):
                out_file.write(json.dumps(
                    {"image_id": item["image_id"], "raw": raw}, ensure_ascii=False
                ) + "\n")

    return output_path


with open("data/splits/test.jsonl", "r", encoding="utf-8") as f:
    test_entries = [json.loads(line) for line in f]
e0_prediction_path = Path("work/e0.jsonl")
e0_evidence_path = Path("report/evidence/e0.json")
if not e0_evidence_path.is_file() and "restore_from_drive" in globals():
    restore_from_drive("E0_test_evidence.json", str(e0_evidence_path))

try:
    current_dataset_version = json.loads(
        Path("data/splits/meta.json").read_text(encoding="utf-8")
    )["dataset_version"]
    cached_evidence = json.loads(e0_evidence_path.read_text(encoding="utf-8"))
    e0_evidence_is_current = (
        cached_evidence.get("score_version") == 1
        and cached_evidence.get("split") == "test"
        and cached_evidence.get("missing_predictions") == 0
        and cached_evidence.get("dataset_version") == current_dataset_version
        and cached_evidence.get("coord") == "norm1000"
    )
except (OSError, ValueError, KeyError):
    e0_evidence_is_current = False

def _e0_predictions_cover_test(path, entries):
    try:
        with path.open(encoding="utf-8") as source:
            rows = [json.loads(line) for line in source if line.strip()]
        expected = {entry["image_id"] for entry in entries}
        actual = [row["image_id"] for row in rows]
        return (
            len(actual) == len(expected)
            and len(set(actual)) == len(actual)
            and set(actual) == expected
            and all(isinstance(row.get("raw"), str) for row in rows)
        )
    except (OSError, ValueError, KeyError, TypeError):
        return False

if not e0_prediction_path.is_file() and "restore_from_drive" in globals():
    restore_from_drive("E0_test_predictions.jsonl", str(e0_prediction_path))
if not e0_evidence_is_current or not _e0_predictions_cover_test(e0_prediction_path, test_entries):
    model = load_base_model()
    model.eval()
    e0_prediction_path = write_predictions_jsonl(
        model,
        test_entries,
        e0_prediction_path,
        SYSTEM_PROMPT,
        batch_size=1,
        max_new_tokens=256,
        desc="Evaluating E0 Zero-shot",
    )
    del model
else:
    print("Dùng lại E0 evidence/predictions cùng dataset_version; bỏ qua inference E0.")
if "backup_to_drive" in globals():
    backup_to_drive(str(e0_prediction_path), "E0_test_predictions.jsonl")
from scripts.score import parse_output
with e0_prediction_path.open(encoding="utf-8") as source:
    e0_predictions = [json.loads(line) for line in source if line.strip()]
e0_raw_examples = []
for row in e0_predictions[:5]:
    parsed = parse_output(row["raw"])
    example = {
        "image_id": row["image_id"], "raw": row["raw"],
        "parse_valid": parsed is not None, "parsed_box_count": len(parsed) if parsed is not None else None,
    }
    e0_raw_examples.append(example)
    print(f"\nE0 raw | image_id={example['image_id']} | parse_valid={example['parse_valid']}\n{example['raw']}")
raw_examples_path = Path("report/evidence/e0_raw_examples.json")
raw_examples_path.parent.mkdir(parents=True, exist_ok=True)
raw_examples_path.write_text(
    json.dumps(e0_raw_examples, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Đã lưu 5 output E0 thô tại {raw_examples_path}")
if "backup_to_drive" in globals():
    backup_to_drive(str(raw_examples_path), "E0_raw_examples.json")
del test_entries
if "model" in globals():
    del model
torch.cuda.empty_cache()


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

subprocess.run([
    sys.executable, "scripts/score.py", "--pred", "work/e0.jsonl",
    "--name", "E0", "--out", "report/evidence/e0.json",
], check=True)
if "backup_to_drive" in globals():
    backup_to_drive("report/evidence/e0.json", "E0_test_evidence.json")

# Diagnostic only: rescore the same norm1000 predictions as if their coordinates were pixels.
wrong_coord_path = Path("report/diagnostics/e0_norm1000_scored_as_pixel.json")
wrong_coord_path.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, "scripts/score.py", "--pred", "work/e0.jsonl",
    "--name", "E0_norm1000_as_pixel", "--split", "test", "--coord", "pixel",
    "--out", str(wrong_coord_path),
], check=True)
correct = json.loads(Path("report/evidence/e0.json").read_text(encoding="utf-8"))
wrong = json.loads(wrong_coord_path.read_text(encoding="utf-8"))
coordinate_diagnostic = {
    "dataset_version": correct["dataset_version"],
    "prediction_file": "work/e0.jsonl",
    "correct_interpretation": {"coord": correct["coord"], "f1": correct["micro"]["f1"], "ci95": correct["f1_ci95"]},
    "mismatched_interpretation": {"coord": wrong["coord"], "f1": wrong["micro"]["f1"], "ci95": wrong["f1_ci95"]},
    "note": "Same E0 predictions scored once with norm1000 and once misread as pixels; diagnostic only, not an E4 ablation.",
}
coordinate_diagnostic_path = Path("report/coordinate_scale_diagnostic.json")
coordinate_diagnostic_path.write_text(json.dumps(coordinate_diagnostic, indent=2), encoding="utf-8")
print(f"Coordinate scale diagnostic saved: {coordinate_diagnostic_path}")
if "backup_to_drive" in globals():
    backup_to_drive(str(wrong_coord_path), "E0_wrong_coordinate_score.json")
    backup_to_drive(str(coordinate_diagnostic_path), "coordinate_scale_diagnostic.json")


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Dataset chat-template có mask loss trên prompt; collator giữ đủ các tensor đa phương thức.
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from PIL import Image


class RoadDamageDataset(Dataset):
    def __init__(self, jsonl_path, processor, coord_scale: str = "norm1000"):
        if coord_scale not in ("norm1000", "pixel"):
            raise ValueError(f"coord_scale không hợp lệ: {coord_scale}")
        with open(jsonl_path, "r", encoding="utf-8") as f:
            self.entries = [json.loads(line) for line in f]
        self.processor = processor
        self.coord_scale = coord_scale

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        item = self.entries[idx]
        with Image.open(Path("data") / item["image_path"]) as source_image:
            image = source_image.convert("RGB")

        answer = objects_to_answer(
            item["objects"],
            coord_scale=self.coord_scale,
            width=item["width"],
            height=item["height"],
        )
        user_message = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": system_prompt_for_scale(self.coord_scale)},
            ],
        }]
        prompt_enc = self.processor.apply_chat_template(
            user_message,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        full_enc = self.processor.apply_chat_template(
            user_message + [{"role": "assistant", "content": [{"type": "text", "text": answer}]}],
            add_generation_prompt=False,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        prompt_ids = prompt_enc["input_ids"]
        full_ids = full_enc["input_ids"]
        prompt_len = prompt_ids.shape[1]
        if full_ids.shape[1] < prompt_len or not torch.equal(full_ids[:, :prompt_len], prompt_ids):
            raise ValueError(
                "Chat template không giữ prompt làm tiền tố của hội thoại đầy đủ; "
                "không thể mask loss bằng prompt_len một cách an toàn."
            )

        labels = full_ids[0].clone()
        labels[:prompt_len] = -100
        attention_mask = full_enc.get("attention_mask")
        if attention_mask is None:
            attention_mask = torch.ones_like(full_ids)

        example = {
            "input_ids": full_ids[0],
            "attention_mask": attention_mask[0].long(),
            "labels": labels,
            "pixel_values": full_enc.get("pixel_values"),
            "image_grid_thw": full_enc.get("image_grid_thw"),
        }
        mm_token_type_ids = full_enc.get("mm_token_type_ids")
        example["mm_token_type_ids"] = mm_token_type_ids[0] if mm_token_type_ids is not None else None
        return example


def collate_fn(batch):
    def pad_1d(key, padding_value):
        values = [item[key] for item in batch]
        if all(value is None for value in values):
            return None
        if any(value is None for value in values):
            raise ValueError(f"Batch có mm field không đồng nhất: {key}")
        return pad_sequence(values, batch_first=True, padding_value=padding_value)

    out = {
        "input_ids": pad_1d("input_ids", processor.tokenizer.pad_token_id),
        "attention_mask": pad_1d("attention_mask", 0),
        "labels": pad_1d("labels", -100),
    }
    mm_token_type_ids = pad_1d("mm_token_type_ids", 0)
    if mm_token_type_ids is not None:
        out["mm_token_type_ids"] = mm_token_type_ids

    for key in ("pixel_values", "image_grid_thw"):
        values = [item[key] for item in batch]
        if all(value is None for value in values):
            continue
        if any(value is None for value in values):
            raise ValueError(f"Batch có mm field không đồng nhất: {key}")
        out[key] = torch.cat(values, dim=0)
    return out

def inspect_training_example(
    split_path: str = "data/splits/train.jsonl",
    output_path: str = "report/training_example_check.json",
):
    """Kiểm tra 220 vision tokens ở 640x360 và decode đúng phần answer sau mask."""
    with open(split_path, encoding="utf-8") as source:
        rows = [json.loads(line) for line in source if line.strip()]
    row = next((item for item in rows if item.get("objects")), rows[0] if rows else None)
    if row is None:
        raise ValueError("Không có ảnh train để kiểm tra token/label.")
    with Image.open(Path("data") / row["image_path"]) as source_image:
        image = source_image.convert("RGB")
    answer = objects_to_answer(
        row["objects"], width=int(row["width"]), height=int(row["height"])
    )
    user = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": SYSTEM_PROMPT},
    ]}]
    prompt_enc = processor.apply_chat_template(
        user, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    )
    full_enc = processor.apply_chat_template(
        user + [{"role": "assistant", "content": [{"type": "text", "text": answer}]}],
        add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors="pt"
    )
    prompt_ids, full_ids = prompt_enc["input_ids"], full_enc["input_ids"]
    prompt_len = prompt_ids.shape[1]
    if not torch.equal(full_ids[:, :prompt_len], prompt_ids):
        raise AssertionError("Prompt không phải tiền tố của chuỗi train đầy đủ.")
    image_token_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
    image_token_count = int((prompt_ids == image_token_id).sum().item())
    labels = full_ids[0].clone()
    labels[:prompt_len] = -100
    target_ids = labels[labels != -100]
    decoded_target = processor.tokenizer.decode(target_ids, skip_special_tokens=False)
    eos_token = processor.tokenizer.eos_token or "<|im_end|>"
    answer_text = decoded_target.removesuffix(eos_token).strip()
    decode_ok = answer_text == answer and decoded_target.rstrip().endswith("<|im_end|>")
    if not decode_ok:
        raise AssertionError(
            "Phần labels != -100 không decode thành đúng JSON + <|im_end|>: "
            f"answer={answer!r}, decoded={decoded_target!r}"
        )
    is_640x360 = int(row["width"]) == 640 and int(row["height"]) == 360
    token_count_ok = (not is_640x360) or image_token_count == 220
    result = {
        "image_id": row["image_id"], "image_width": int(row["width"]),
        "image_height": int(row["height"]), "image_tokens": image_token_count,
        "expected_image_tokens": 220 if is_640x360 else None,
        "image_token_check": "PASS" if token_count_ok else "WARN",
        "prompt_prefix_check": "PASS", "answer_decode_check": "PASS",
        "answer": answer, "decoded_masked_target": decoded_target,
    }
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(
        f"Training example {row['image_id']}: image_tokens={image_token_count}"
        f"{' (expected 220)' if is_640x360 else ''}; "
        f"masked target decode={'PASS' if decode_ok else 'FAIL'}"
    )
    if not token_count_ok:
        raise AssertionError(
            f"Ảnh 640x360 phải có đúng 220 image_pad tokens; nhận được {image_token_count}. "
            "Dừng trước khi train để kiểm tra processor/runtime."
        )
    print(f"Đã lưu kiểm tra token/label tại {output}")
    if "backup_to_drive" in globals():
        backup_to_drive(str(output), "training_example_check.json")
    return result


training_example_check = inspect_training_example()


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re
from math import prod

import torch
from peft import LoraConfig, get_peft_model

PROJECTOR_NAME_PARTS = ("visual.merger", "visual.deepstack_merger_list")


def _is_projector_parameter(name: str) -> bool:
    return any(part in name for part in PROJECTOR_NAME_PARTS)


def _parameter_logical_numel(parameter) -> int:
    """Dùng shape gốc của BitsAndBytes params thay vì kích thước packed 4-bit."""
    quant_state = getattr(parameter, "quant_state", None)
    logical_shape = getattr(quant_state, "shape", None)
    return int(prod(logical_shape)) if logical_shape is not None else int(parameter.numel())


def _architecture_parameter_counts(model) -> dict:
    """Đếm tham số gốc theo khối, loại LoRA nếu hàm được gọi trên PEFT model."""
    language_model = lm_head = visual = projector = total = 0
    for name, parameter in model.named_parameters():
        if "lora_" in name:
            continue
        count = _parameter_logical_numel(parameter)
        total += count
        if "language_model." in name:
            language_model += count
        if "lm_head" in name:
            lm_head += count
        if "visual." in name:
            visual += count
        if _is_projector_parameter(name):
            projector += count
    return {
        "language_model": int(language_model),
        "lm_head": int(lm_head),
        "language_model_plus_lm_head": int(language_model + lm_head),
        "visual_total_including_projector": int(visual),
        "visual_core_excluding_projector": int(visual - projector),
        "projector_merger_and_deepstack": int(projector),
        "total_base_model": int(total),
    }


def apply_setup(
    model,
    setup: str,
    lora_r: int = 16,
    lora_alpha: int = 32,
    seed: int = 42,
):
    """Đóng băng/mở đúng các khối theo E1, E2 hoặc E3 và xác nhận target count."""
    setup_upper = setup.upper()
    if setup_upper not in ("E1", "E2", "E3"):
        raise ValueError(f"Setup không hợp lệ: {setup}. Hỗ trợ: 'E1', 'E2', 'E3'.")
    if lora_r < 1 or lora_alpha < 1:
        raise ValueError("lora_r và lora_alpha phải là số nguyên dương")

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.enable_input_require_grads()

    llm_targets = (
        r"language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|"
        r"mlp\.(gate|up|down)_proj)"
    )
    if setup_upper == "E3":
        target_modules = (
            r".*(language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|"
            r"mlp\.(gate|up|down)_proj)|visual\.blocks\.\d+\.(attn\.(qkv|proj)|"
            r"mlp\.(linear_fc1|linear_fc2)))"
        )
        expected_modules = 348
    else:
        target_modules = r".*" + llm_targets
        expected_modules = 252

    matched_modules = [
        name for name, _ in model.named_modules()
        if re.fullmatch(target_modules, name)
    ]
    if len(matched_modules) != expected_modules:
        raise RuntimeError(
            f"{setup_upper}: regex khớp {len(matched_modules)} module, "
            f"mong đợi {expected_modules}; kiểm tra lại cây module Qwen3-VL."
        )

    peft_model = get_peft_model(
        model,
        LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=0.05,
            target_modules=target_modules,
            bias="none",
            task_type="CAUSAL_LM",
        ),
    )
    lora_params = sum(
        parameter.numel()
        for name, parameter in peft_model.named_parameters()
        if "lora_" in name and parameter.requires_grad
    )
    if lora_params == 0:
        raise RuntimeError(f"{setup_upper}: PEFT không tạo tham số LoRA trainable.")

    if setup_upper in ("E2", "E3"):
        opened_params = 0
        for name, parameter in peft_model.named_parameters():
            if _is_projector_parameter(name):
                parameter.data = parameter.data.to(torch.float32)
                parameter.requires_grad = True
                opened_params += parameter.numel()
        if opened_params == 0:
            raise RuntimeError(f"{setup_upper}: không tìm thấy projector cần mở.")
        print(f"[{setup_upper}] Projector trainable: {opened_params:,} tham số ở float32.")

    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in peft_model.parameters())
    print(
        f"[{setup_upper}] Khớp {len(matched_modules)} module LoRA; "
        f"trainable {trainable_params:,}/{total_params:,} ({100 * trainable_params / total_params:.3f}%)."
    )
    peft_model._lab_setup_metadata = {
        "setup": setup_upper,
        "lora_rank": int(lora_r),
        "lora_alpha": int(lora_alpha),
        "lora_target_module_count": len(matched_modules),
        "llm_lora_module_count": 252,
        "vit_lora_module_count": 96 if setup_upper == "E3" else 0,
        "projector_trainable": setup_upper in ("E2", "E3"),
        "projector_trainable_dtype": "float32" if setup_upper in ("E2", "E3") else "frozen",
        "quantization": "4-bit NF4; visual and lm_head excluded",
        "architecture_params": _architecture_parameter_counts(model),
    }
    return peft_model

## 6. Vòng huấn luyện

In [ ]:
import json
import math
import time
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup


def train_setup(
    peft_model,
    setup_name: str,
    max_steps: int = 300,
    batch_size: int = 1,
    grad_accum_steps: int = 8,
    coord_scale: str = "norm1000",
    train_fraction: float = 1.0,
    seed: int = 42,
):
    """Huấn luyện SFT theo GUIDE; hỗ trợ subset dữ liệu và thang pixel cho E4."""
    if not torch.cuda.is_available():
        raise RuntimeError("Huấn luyện yêu cầu GPU CUDA; hãy chọn runtime T4 trên Colab.")
    if max_steps < 1 or batch_size < 1 or grad_accum_steps < 1:
        raise ValueError("max_steps, batch_size và grad_accum_steps phải lớn hơn 0")
    if not 0 < train_fraction <= 1:
        raise ValueError("train_fraction phải thuộc (0, 1]")

    setup_name = setup_name.upper()
    train_dataset = RoadDamageDataset(
        "data/splits/train.jsonl", processor, coord_scale=coord_scale
    )
    full_train_size = len(train_dataset)
    if train_fraction < 1.0:
        keep = max(1, math.ceil(len(train_dataset) * train_fraction))
        generator = torch.Generator().manual_seed(seed)
        indices = torch.randperm(len(train_dataset), generator=generator)[:keep].tolist()
        train_dataset = Subset(train_dataset, indices)
        print(f"Dùng {keep}/{full_train_size} ảnh train.")

    loader_generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        generator=loader_generator,
        num_workers=0,
    )
    if len(train_loader) == 0:
        raise ValueError("Tập train rỗng")

    trainable_parameters = [p for p in peft_model.parameters() if p.requires_grad]
    trainable_count = sum(p.numel() for p in trainable_parameters)
    if trainable_count == 0:
        raise RuntimeError("Không có tham số trainable; hãy gọi apply_setup trước khi train.")

    optimizer = AdamW(trainable_parameters, lr=2e-4, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(max_steps * 0.05)),
        num_training_steps=max_steps,
    )
    scaler = torch.amp.GradScaler("cuda")

    peft_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    peft_model.config.use_cache = False
    peft_model.train()
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    step = 0
    loss_history = []
    loader_iter = iter(train_loader)
    progress = tqdm(total=max_steps, desc=f"Huấn luyện {setup_name}")
    while step < max_steps:
        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0

        for _ in range(grad_accum_steps):
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(train_loader)
                batch = next(loader_iter)

            batch = {
                key: value.to("cuda") if isinstance(value, torch.Tensor) else value
                for key, value in batch.items()
            }
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = peft_model(**batch).loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn (NaN/Inf) tại step {step}; dừng train.")

            scaler.scale(loss / grad_accum_steps).backward()
            step_loss += float(loss.detach().item()) / grad_accum_steps

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        loss_history.append({"step": step + 1, "loss": float(step_loss)})
        step += 1
        progress.update(1)
        progress.set_postfix({"loss": f"{step_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

        if step % 100 == 0:
            checkpoint_path = Path(f"work/{setup_name}_adapter_step{step}")
            checkpoint_path.mkdir(parents=True, exist_ok=True)
            peft_model.save_pretrained(checkpoint_path)
            checkpoint_projector = {
                name: parameter.detach().cpu().clone()
                for name, parameter in peft_model.named_parameters()
                if _is_projector_parameter(name) and parameter.requires_grad
            }
            if checkpoint_projector:
                torch.save(checkpoint_projector, checkpoint_path / "projector_weights.pt")
            loss_path = Path(f"work/{setup_name.lower()}_loss.json")
            loss_path.parent.mkdir(parents=True, exist_ok=True)
            loss_path.write_text(json.dumps(loss_history, indent=2), encoding="utf-8")
            if "backup_to_drive" in globals():
                backup_to_drive(str(checkpoint_path), f"{setup_name}_adapter_step{step}")
                backup_to_drive(str(loss_path), f"{setup_name}_loss.json")

    progress.close()
    torch.cuda.synchronize()
    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    save_path = Path(f"work/{setup_name}_adapter")
    save_path.mkdir(parents=True, exist_ok=True)
    peft_model.save_pretrained(save_path)
    projector_weights = {
        name: parameter.detach().cpu().clone()
        for name, parameter in peft_model.named_parameters()
        if _is_projector_parameter(name) and parameter.requires_grad
    }
    if projector_weights:
        torch.save(projector_weights, save_path / "projector_weights.pt")

    dataset_meta_path = Path("data/splits/meta.json")
    if not dataset_meta_path.is_file():
        raise FileNotFoundError("Thiếu data/splits/meta.json; chạy prepare_data.py trước khi train.")
    dataset_version = json.loads(dataset_meta_path.read_text(encoding="utf-8"))["dataset_version"]
    meta_info = {
        "dataset_version": dataset_version,
        "loss_logging": "mean_per_optimizer_step",
        "training_variant": setup_name,
        "setup_details": dict(getattr(peft_model, "_lab_setup_metadata", {})),
        "architecture_params": dict(getattr(peft_model, "_lab_setup_metadata", {}).get("architecture_params", {})),
        "training_config": {
            "optimizer": "AdamW", "learning_rate": 2.0e-4,
            "micro_batch_size": batch_size, "grad_accum_steps": grad_accum_steps,
            "effective_batch_size": batch_size * grad_accum_steps,
        },
        "trainable_params": int(trainable_count),
        "peak_vram_gb": round(float(peak_vram_gb), 2),
        "train_minutes": round(float(train_minutes), 2),
        "coord_scale": coord_scale,
        "train_fraction": float(train_fraction),
        "max_steps": int(max_steps),
        "projector_saved": bool(projector_weights),
        "loss_steps": len(loss_history),
        "loss_first": loss_history[0]["loss"],
        "loss_last": loss_history[-1]["loss"],
        "loss_log": f"work/{setup_name.lower()}_loss.json",
    }
    loss_path = Path(meta_info["loss_log"])
    loss_path.parent.mkdir(parents=True, exist_ok=True)
    loss_path.write_text(json.dumps(loss_history, indent=2), encoding="utf-8")
    run_file = Path(f"work/run_{setup_name.lower()}.json")
    run_file.parent.mkdir(parents=True, exist_ok=True)
    run_file.write_text(json.dumps(meta_info, indent=2), encoding="utf-8")
    print(f"Đã lưu adapter tại {save_path}; metadata: {meta_info}")

    if "backup_to_drive" in globals():
        backup_to_drive(str(save_path), f"{setup_name}_adapter")
        backup_to_drive(str(run_file), f"{setup_name}_run.json")
        backup_to_drive(str(loss_path), f"{setup_name}_loss.json")
    return meta_info


def train_experiment(
    setup_name: str,
    max_steps: int = 300,
    batch_size: int = 1,
    grad_accum_steps: int = 8,
):
    """Nạp model gốc mới, áp dụng E1/E2/E3, train rồi giải phóng VRAM."""
    setup_name = setup_name.upper()
    if setup_name not in ("E1", "E2", "E3"):
        raise ValueError("setup_name phải là E1, E2 hoặc E3")
    base_model = load_base_model()
    peft_model = apply_setup(base_model, setup_name, seed=42)
    meta = train_setup(
        peft_model,
        setup_name,
        max_steps=max_steps,
        batch_size=batch_size,
        grad_accum_steps=grad_accum_steps,
        seed=42,
    )
    del peft_model, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return meta

# Ví dụ: train_experiment("E1"), sau đó run_eval("E1", meta, split="val") để gỡ lỗi.
# Chỉ gọi run_eval(..., split="test") khi sẵn sàng chấm test.

## 7. Suy luận và chấm E1–E3

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import torch
from PIL import Image
from peft import PeftModel


def _load_projector_weights(model, projector_path: Path, required: bool = False) -> bool:
    if not projector_path.exists():
        if required:
            raise FileNotFoundError(
                f"Thiếu projector_weights.pt tại {projector_path}; kết quả sẽ không khớp setup đã train."
            )
        return False

    # Phục hồi dtype float32 đã dùng khi train projector E2/E3. Inference chạy trong autocast fp16.
    for name, parameter in model.named_parameters():
        if _is_projector_parameter(name):
            parameter.data = parameter.data.to(torch.float32)
    weights = torch.load(projector_path, map_location="cpu")
    if not isinstance(weights, dict) or not weights:
        raise ValueError(f"Checkpoint projector rỗng hoặc sai định dạng: {projector_path}")
    if not all(_is_projector_parameter(name) for name in weights):
        raise ValueError("Checkpoint projector chứa key ngoài visual.merger/deepstack_merger_list.")

    expected_keys = {
        name for name, _ in model.named_parameters() if _is_projector_parameter(name)
    }
    if set(weights) != expected_keys:
        missing = sorted(expected_keys - set(weights))
        extra = sorted(set(weights) - expected_keys)
        raise RuntimeError(
            f"Projector checkpoint không đầy đủ: thiếu={missing[:5]}, ngoài dự kiến={extra[:5]}"
        )
    result = model.load_state_dict(weights, strict=False)
    if result.unexpected_keys:
        raise RuntimeError(f"Projector keys không được nạp: {result.unexpected_keys[:5]}")
    print(f"Đã nạp và xác nhận {len(weights)} tensors projector từ {projector_path}.")
    return True


def run_eval(
    setup_name: str,
    meta_info: dict | None = None,
    split: str = "test",
    batch_size: int = 1,
    coord_scale: str = "norm1000",
):
    """Nạp adapter/projector, sinh dự đoán theo batch và gọi trọng tài scripts/score.py."""
    setup_name = setup_name.upper()
    if split not in ("test", "val"):
        raise ValueError("split chỉ nhận 'test' hoặc 'val'")
    if coord_scale not in ("norm1000", "pixel"):
        raise ValueError(f"coord_scale không hợp lệ: {coord_scale}")

    adapter_path = Path(f"work/{setup_name}_adapter")
    if not adapter_path.exists():
        raise FileNotFoundError(f"Chưa tìm thấy adapter tại {adapter_path}; hãy train trước.")
    if meta_info is None:
        run_file = Path(f"work/run_{setup_name.lower()}.json")
        if run_file.exists():
            meta_info = json.loads(run_file.read_text(encoding="utf-8"))

    print(f"--- ĐÁNH GIÁ {setup_name} trên split={split}, coord={coord_scale} ---")
    base_model = load_base_model()
    eval_model = PeftModel.from_pretrained(base_model, str(adapter_path))
    projector_path = adapter_path / "projector_weights.pt"
    projector_required = setup_name in ("E2", "E3") or bool(
        meta_info and meta_info.get("projector_saved")
    )
    _load_projector_weights(eval_model, projector_path, required=projector_required)
    eval_model.eval()

    split_file = Path(f"data/splits/{split}.jsonl")
    with split_file.open("r", encoding="utf-8") as f:
        entries = [json.loads(line) for line in f]
    prediction_name = (f"{setup_name.lower()}.jsonl" if split == "test"
                       else f"val_{setup_name.lower()}.jsonl")
    prediction_path = Path("work") / prediction_name
    write_predictions_jsonl(
        eval_model,
        entries,
        prediction_path,
        system_prompt_for_scale(coord_scale),
        batch_size=batch_size,
        desc=f"Suy luận {setup_name} ({split})",
    )

    del eval_model, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    evidence_name = (f"{setup_name.lower()}.json" if split == "test"
                     else f"debug_{setup_name.lower()}_val.json")
    evidence_out = Path("report/evidence") / evidence_name
    evidence_out.parent.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        "scripts/score.py",
        "--pred", str(prediction_path),
        "--name", setup_name,
        "--split", split,
        "--coord", coord_scale,
        "--out", str(evidence_out),
    ]
    if meta_info:
        for key in ("trainable_params", "peak_vram_gb", "train_minutes"):
            if key in meta_info:
                command.extend(["--meta", f"{key}={meta_info[key]}"])

    subprocess.run(command, check=True)
    if "backup_to_drive" in globals():
        backup_to_drive(str(prediction_path), f"{setup_name}_{split}_predictions.jsonl")
        backup_to_drive(str(evidence_out), f"{setup_name}_{split}_evidence.json")
    print(f"Đã chấm điểm xong {setup_name}; bằng chứng tại {evidence_out}")
    return evidence_out

## 8. E4: Thí nghiệm Ablation (Cô lập một biến số)


In [ ]:
# E4: mỗi biến thể được train lại từ model gốc với cùng recipe; chỉ thay biến đang khảo sát.
# Mặc định chọn E1 làm setup đối chứng. Đổi base_setup_name thành E2/E3 nếu đó là setup tốt nhất.


def _run_e4_variant(
    base_setup_name: str,
    run_name: str,
    max_steps: int = 300,
    lora_rank: int = 16,
    coord_scale: str = "norm1000",
    train_fraction: float = 1.0,
):
    base_setup_name = base_setup_name.upper()
    if base_setup_name not in ("E1", "E2", "E3"):
        raise ValueError("base_setup_name phải là E1, E2 hoặc E3")

    base_model = load_base_model()
    peft_model = apply_setup(
        base_model,
        base_setup_name,
        lora_r=lora_rank,
        lora_alpha=2 * lora_rank,
    )
    meta = train_setup(
        peft_model,
        run_name,
        max_steps=max_steps,
        coord_scale=coord_scale,
        train_fraction=train_fraction,
    )
    meta.update({
        "base_setup_name": base_setup_name,
        "lora_rank": int(lora_rank),
        "coord_scale": coord_scale,
        "train_fraction": float(train_fraction),
    })
    run_file = Path(f"work/run_{run_name.lower()}.json")
    run_file.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    if "backup_to_drive" in globals():
        backup_to_drive(str(run_file), f"{run_name}_run.json")
    del peft_model, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return run_eval(
        run_name,
        meta_info=meta,
        split="test",
        coord_scale=coord_scale,
    )


def run_e4_ablation_pixel(base_setup_name: str = "E1", max_steps: int = 300):
    """Train E4 pixel: pixel trong nhãn, prompt và trọng tài; so với base_setup norm1000."""
    return _run_e4_variant(
        base_setup_name=base_setup_name,
        run_name="E4_coord_scale_pixel",
        max_steps=max_steps,
        lora_rank=16,
        coord_scale="pixel",
    )


def run_e4_ablation_rank(rank: int, base_setup_name: str = "E1", max_steps: int = 300):
    """Train một giá trị rank trong cấu hình E4; alpha luôn bằng 2*r theo configs/e4.yaml."""
    if rank not in (4, 64):
        raise ValueError("configs/e4.yaml yêu cầu rank=4 hoặc rank=64")
    return _run_e4_variant(
        base_setup_name=base_setup_name,
        run_name=f"E4_rank_{rank}",
        max_steps=max_steps,
        lora_rank=rank,
        coord_scale="norm1000",
    )


def run_e4_rank_pair(base_setup_name: str = "E1", max_steps: int = 300):
    """Chạy đủ r=4 và r=64; so sánh với evidence của base_setup_name (r=16)."""
    results = {}
    for rank in (4, 64):
        results[rank] = run_e4_ablation_rank(
            rank=rank,
            base_setup_name=base_setup_name,
            max_steps=max_steps,
        )
    return results


# Chọn đúng một biến cho E4 và chạy các giá trị tương ứng:
# run_e4_ablation_pixel(base_setup_name="E1", max_steps=300)
# run_e4_rank_pair(base_setup_name="E1", max_steps=300)
# run_e4_data_size_pair(base_setup_name="E1", max_steps=300)


def run_e4_ablation_data_size(
    train_fraction: float,
    base_setup_name: str = "E1",
    max_steps: int = 300,
):
    """Train một mức data_size trong configs/e4.yaml; chỉ đổi lượng ảnh train."""
    if train_fraction not in (0.1, 1.0):
        raise ValueError("configs/e4.yaml yêu cầu train_fraction=0.1 hoặc 1.0")
    percent = int(round(train_fraction * 100))
    return _run_e4_variant(
        base_setup_name=base_setup_name,
        run_name=f"E4_data_size_{percent}",
        max_steps=max_steps,
        lora_rank=16,
        coord_scale="norm1000",
        train_fraction=train_fraction,
    )


def run_e4_data_size_pair(base_setup_name: str = "E1", max_steps: int = 300):
    """Chạy 10% và 100% data_size với cùng setup/hyperparameters."""
    return {
        fraction: run_e4_ablation_data_size(
            train_fraction=fraction,
            base_setup_name=base_setup_name,
            max_steps=max_steps,
        )
        for fraction in (0.1, 1.0)
    }

## 9. E5 (Tuỳ chọn, +10 điểm): GRPO với Reward IoU

> E5 nạp adapter tốt nhất từ E1–E3, giữ projector đã học nếu có, rồi cập nhật LoRA bằng GRPO. Trong [tài liệu TRL 1.9.2 về VLM training](https://github.com/huggingface/trl/blob/v1.9.2/docs/source/grpo_trainer.md#vision-language-model-vlm-training), Qwen3-VL chưa nằm trong danh sách VLM được nêu là đã kiểm thử; hãy kiểm tra một bước train trên Colab trước khi chạy đủ 60 bước.

In [ ]:
# E5: reward IoU thuần Python, kiểm tra trước khi khởi tạo GRPOTrainer.
import json
import random
from pathlib import Path

import torch
from scripts.score import iou_xyxy, parse_output


def compute_iou_reward(completion: str, gt_boxes: list[dict]) -> float:
    """Reward theo GUIDE: JSON hỏng=0; rỗng+rỗng=1; ghép cùng lớp với IoU>=0.5."""
    pred_boxes = parse_output(completion)
    if pred_boxes is None:
        return 0.0
    if not gt_boxes and not pred_boxes:
        return 1.0

    pairs = []
    for gt_index, gt in enumerate(gt_boxes):
        gt_bbox = gt.get("bbox", gt.get("bbox_2d"))
        for pred_index, pred in enumerate(pred_boxes):
            if gt.get("label") != pred.get("label"):
                continue
            pred_bbox = pred.get("bbox", pred.get("bbox_2d"))
            overlap = iou_xyxy(gt_bbox, pred_bbox)
            if overlap >= 0.5:
                pairs.append((overlap, gt_index, pred_index))

    # Tie-break ổn định để cùng input luôn cho cùng matching.
    pairs.sort(key=lambda pair: (-pair[0], pair[1], pair[2]))
    used_gt, used_pred = set(), set()
    total_iou = 0.0
    for overlap, gt_index, pred_index in pairs:
        if gt_index not in used_gt and pred_index not in used_pred:
            used_gt.add(gt_index)
            used_pred.add(pred_index)
            total_iou += overlap

    denominator = max(len(gt_boxes), len(pred_boxes), 1)
    return round(float(0.2 + 0.8 * total_iou / denominator), 4)


def _completion_to_text(completion) -> str:
    """Lấy nội dung assistant từ dạng chuỗi hoặc conversational completion của TRL."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        messages = [completion]
    elif isinstance(completion, (list, tuple)):
        messages = list(completion)
    else:
        return ""

    assistant_messages = [
        message for message in messages
        if isinstance(message, dict) and message.get("role") == "assistant"
    ]
    message = assistant_messages[-1] if assistant_messages else (messages[-1] if messages else {})
    content = message.get("content", "") if isinstance(message, dict) else ""
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        return "".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict) and isinstance(block.get("text", ""), str)
        )
    return ""


def test_compute_iou_reward():
    gt = [{"label": "pothole", "bbox": [100, 100, 200, 200]}]
    perfect = '[{"label":"pothole","bbox_2d":[100,100,200,200]}]'
    malformed = "Here is pothole at [100, 100]"
    wrong_class = '[{"label":"crack","bbox_2d":[100,100,200,200]}]'
    extra_box = (
        '[{"label":"pothole","bbox_2d":[100,100,200,200]},'
        '{"label":"pothole","bbox_2d":[500,500,600,600]}]'
    )

    assert compute_iou_reward(perfect, gt) == 1.0
    assert compute_iou_reward(malformed, gt) == 0.0
    assert compute_iou_reward(wrong_class, gt) == 0.2
    assert compute_iou_reward("[]", []) == 1.0
    assert 0.2 < compute_iou_reward(extra_box, gt) < 1.0
    assert _completion_to_text([{"role": "assistant", "content": perfect}]) == perfect
    print("Tất cả kiểm tra reward IoU đã qua.")


test_compute_iou_reward()


def train_and_eval_e5_grpo(best_adapter_name: str = "E1", max_steps: int = 60):
    """Train GRPO 60 bước từ adapter tốt nhất, lưu adapter/projector rồi chấm test."""
    if not torch.cuda.is_available():
        raise RuntimeError("E5 yêu cầu GPU CUDA; hãy chạy trong Colab T4.")
    if max_steps < 1:
        raise ValueError("max_steps phải lớn hơn 0")

    from datasets import Dataset, Image as HFImage
    from peft import PeftModel
    from trl import GRPOConfig, GRPOTrainer
    import time

    best_adapter_name = best_adapter_name.upper()
    if best_adapter_name not in ("E1", "E2", "E3"):
        raise ValueError("best_adapter_name phải là E1, E2 hoặc E3")
    adapter_src = Path(f"work/{best_adapter_name}_adapter")
    if not adapter_src.exists():
        raise FileNotFoundError(f"Không tìm thấy adapter khởi tạo: {adapter_src}")

    base_model = load_base_model()
    train_model = PeftModel.from_pretrained(
        base_model,
        str(adapter_src),
        is_trainable=True,
    )
    source_projector = adapter_src / "projector_weights.pt"
    projector_was_saved = _load_projector_weights(
        train_model,
        source_projector,
        required=best_adapter_name in ("E2", "E3"),
    )
    if projector_was_saved:
        # E2/E3 đã train projector; tiếp tục cập nhật nó trong GRPO và lưu lại.
        for name, parameter in train_model.named_parameters():
            if _is_projector_parameter(name):
                parameter.requires_grad = True

    with open("data/splits/train.jsonl", "r", encoding="utf-8") as f:
        all_train_rows = [json.loads(line) for line in f]
    rng = random.Random(42)
    train_rows = rng.sample(all_train_rows, k=min(200, len(all_train_rows)))
    grpo_rows = []
    for item in train_rows:
        image_path = Path("data") / item["image_path"]
        if not image_path.is_file():
            raise FileNotFoundError(f"Thiếu ảnh GRPO: {image_path}")
        grpo_rows.append({
            # TRL 1.9.2 gắn cột image vào prompt hội thoại khi tạo batch.
            "prompt": [{"role": "user", "content": SYSTEM_PROMPT}],
            "image": str(image_path),
            "gt_boxes": [yolo_to_box(obj["label"], obj["yolo"]) for obj in item["objects"]],
        })
    if not grpo_rows:
        raise ValueError("Không có ảnh train để tạo tập GRPO")
    grpo_dataset = Dataset.from_list(grpo_rows).cast_column("image", HFImage())

    def reward_function(completions, gt_boxes, **kwargs):
        if len(completions) != len(gt_boxes):
            raise ValueError(
                f"TRL truyền {len(completions)} completions nhưng {len(gt_boxes)} ground truths"
            )
        return [
            compute_iou_reward(_completion_to_text(completion), gt)
            for completion, gt in zip(completions, gt_boxes, strict=True)
        ]

    output_dir = Path("work/E5_adapter")
    grpo_config = GRPOConfig(
        output_dir=str(output_dir),
        learning_rate=5.0e-6,
        num_generations=4,
        max_completion_length=128,
        max_steps=max_steps,
        beta=0.0,
        per_device_train_batch_size=1,
        # Effective batch=4 để chia hết cho num_generations=4 theo TRL 1.9.2.
        gradient_accumulation_steps=4,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=5,
        save_strategy="no",
        remove_unused_columns=False,
        use_vllm=False,
        report_to="none",
        dataloader_num_workers=0,
    )
    trainer = GRPOTrainer(
        model=train_model,
        reward_funcs=reward_function,
        args=grpo_config,
        train_dataset=grpo_dataset,
        processing_class=processor,
    )

    trainable_count = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    if trainable_count == 0:
        raise RuntimeError("GRPOTrainer không nhận được tham số trainable; dừng để tránh chạy rỗng.")
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    trainer.train()
    training_log = trainer.state.log_history
    log_file = Path("work/e5_grpo_log.json")
    log_file.parent.mkdir(parents=True, exist_ok=True)
    log_file.write_text(
        json.dumps(training_log, indent=2, default=lambda value: value.item()
                   if hasattr(value, "item") else str(value)),
        encoding="utf-8",
    )
    torch.cuda.synchronize()
    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(output_dir))
    projector_weights = {
        name: parameter.detach().cpu().clone()
        for name, parameter in trainer.model.named_parameters()
        if _is_projector_parameter(name) and parameter.requires_grad
    }
    if projector_weights:
        torch.save(projector_weights, output_dir / "projector_weights.pt")

    meta = {
        "trainable_params": int(trainable_count),
        "peak_vram_gb": round(float(peak_vram_gb), 2),
        "train_minutes": round(float(train_minutes), 2),
        "projector_saved": bool(projector_weights),
        "projector_loaded_from": best_adapter_name if projector_was_saved else None,
        "training_log": str(log_file),
        "log_entries": len(training_log),
    }
    run_file = Path("work/run_e5.json")
    run_file.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    del trainer, train_model, base_model
    torch.cuda.empty_cache()
    if "backup_to_drive" in globals():
        backup_to_drive(str(output_dir), "E5_adapter")
        backup_to_drive(str(run_file), "E5_run.json")
        backup_to_drive(str(log_file), "E5_training_log.json")

    return run_eval("E5", meta_info=meta, split="test", batch_size=1)


print("Reward IoU đã sẵn sàng; train_and_eval_e5_grpo() sẽ train và chấm E5 khi được gọi.")

## 10. Chạy nghiệm thu E1–E4 và xem kết quả

Khi bấm **Run all**, cell cuối sẽ tự train các setup E1/E2/E3 (mỗi setup 300 bước nếu chưa có adapter đúng cấu hình), chọn setup nền theo validation, chạy cả hai mức rank E4 (4 và 64), chấm test, tạo sáu ảnh GT/prediction có chú thích, đồ thị loss và so sánh CI95 của E4, rồi in bảng metrics và kết quả `scripts/check_submission.py`. Notebook cũng lưu split audit, kiểm tra round-trip tọa độ, 5 raw output E0, kiểm tra 220 image tokens/answer mask, và tự điền bảng số liệu cùng dataset_version vào `REPORT.md`. Đây là lượt chạy dài theo GUIDE; chọn GPU T4 và giữ Google Drive đã mount để lưu adapter, log loss, evidence và báo cáo.

Có thể đổi `ACCEPTANCE_MAX_STEPS` hoặc `ACCEPTANCE_E4_VARIABLE` trong cell cuối. E4 hỗ trợ `rank`, `data_size` (10%/100%) hoặc `coord_scale` (norm1000/pixel). E5 là bonus, mặc định tắt; bật `ACCEPTANCE_RUN_E5 = True` sau khi đã xác nhận GRPO tương thích với runtime.

Muốn chỉ xem trạng thái artifact hiện có mà không train, chạy `run_lab_acceptance()`. Bảng và checker được lưu tại `report/acceptance_summary.md`; `REPORT.md` vẫn cần phần trả lời phân tích dựa trên số liệu thật. Sau khi nghiệm thu, tải ZIP tại `/content/K4-Track-4-Day10-VisionLanguageModel/report/day10_report_handoff.zip` qua Files → `report/` → Download, hoặc lấy bản sao trong `MyDrive/AI20K_Lab_Day10_Checkpoints/day10_report_handoff.zip`. ZIP giữ nguyên `REPORT.md`, `report/` và `work/`; giải nén ZIP tại repo root. Nếu summary ghi `BLOCKED`/`FAIL` hoặc có lỗi training, ZIP có thể chỉ chứa kết quả một phần; cần kiểm tra status trước khi dùng làm kết luận REPORT.


In [ ]:
import json
import math
import re
import subprocess
import sys
from pathlib import Path


def _expected_dataset_version():
    meta_path = Path("data/splits/meta.json")
    if not meta_path.is_file():
        raise FileNotFoundError(f"Thiếu {meta_path}; cần chuẩn bị dataset trước khi nghiệm thu.")
    return json.loads(meta_path.read_text(encoding="utf-8"))["dataset_version"]


def _load_valid_evidence(
    name: str,
    split: str = "test",
    meta_info: dict | None = None,
    coord_scale: str | None = None,
):
    lower_name = name.lower()
    filename = f"{lower_name}.json" if split == "test" else f"debug_{lower_name}_val.json"
    path = Path("report/evidence") / filename
    if not path.is_file() and "restore_from_drive" in globals():
        restore_from_drive(f"{name.upper()}_{split}_evidence.json", str(path))
    if not path.is_file():
        return None
    result = json.loads(path.read_text(encoding="utf-8"))
    expected_version = _expected_dataset_version()
    problems = []
    if result.get("score_version") != 1:
        problems.append("score_version != 1")
    if result.get("split") != split:
        problems.append(f"split != {split}")
    if result.get("missing_predictions") != 0:
        problems.append(f"missing_predictions={result.get('missing_predictions')}")
    if result.get("dataset_version") != expected_version:
        problems.append("dataset_version không khớp data/splits/meta.json")
    if coord_scale is not None and result.get("coord") != coord_scale:
        problems.append(f"coord != {coord_scale}")
    if problems:
        raise ValueError(f"Evidence {path} không được tái sử dụng: {'; '.join(problems)}")
    if meta_info is not None:
        evidence_meta = result.get("meta", {})
        for key in ("trainable_params", "peak_vram_gb", "train_minutes"):
            if key not in meta_info:
                continue
            actual, expected = evidence_meta.get(key), meta_info[key]
            if isinstance(actual, (int, float)) and isinstance(expected, (int, float)):
                matches = math.isclose(actual, expected, rel_tol=1e-6, abs_tol=1e-6)
            else:
                matches = actual == expected
            if not matches:
                print(f"Evidence {path} cũ hoặc khác run metadata ({key}); sẽ chấm lại.")
                return None
    return result


def _load_or_train_adapter(name: str, max_steps: int):
    adapter_path = Path(f"work/{name}_adapter")
    run_path = Path(f"work/run_{name.lower()}.json")
    if "restore_from_drive" in globals():
        if not adapter_path.is_dir():
            restore_from_drive(f"{name}_adapter", str(adapter_path))
        if not run_path.is_file():
            restore_from_drive(f"{name}_run.json", str(run_path))
    if adapter_path.is_dir() and run_path.is_file():
        meta = json.loads(run_path.read_text(encoding="utf-8"))
        has_adapter_weights = any(
            (adapter_path / filename).is_file()
            for filename in ("adapter_model.safetensors", "adapter_model.bin")
        )
        projector_ok = (
            name not in ("E2", "E3")
            or (meta.get("projector_saved") is True
                and (adapter_path / "projector_weights.pt").is_file())
        )
        current_dataset_version = _expected_dataset_version()
        recipe_ok = (
            meta.get("max_steps") == max_steps
            and meta.get("dataset_version") == current_dataset_version
            and meta.get("loss_logging") == "mean_per_optimizer_step"
        )
        if recipe_ok and has_adapter_weights and projector_ok:
            print(f"Dùng lại adapter đã lưu: {adapter_path}")
            return meta
        print(
            f"Adapter {name} thiếu weight/projector hoặc có max_steps={meta.get('max_steps')}; "
            f"train lại với {max_steps} bước."
        )
    return train_experiment(name, max_steps=max_steps)


def _best_setup_from_validation():
    validation = {name: _load_valid_evidence(name, "val") for name in ("E1", "E2", "E3")}
    missing = [name for name, evidence in validation.items() if evidence is None]
    if missing:
        raise RuntimeError(f"Thiếu validation evidence cho {missing}; chạy suite trước.")
    return max(validation, key=lambda name: validation[name]["micro"]["f1"])


def _e4_result_reusable(
    run_name: str,
    max_steps: int,
    base_setup_name: str,
    lora_rank: int,
    coord_scale: str,
    train_fraction: float,
):
    try:
        evidence = _load_valid_evidence(run_name, "test", coord_scale=coord_scale)
    except ValueError:
        return False
    if evidence is None:
        return False
    run_path = Path(f"work/run_{run_name.lower()}.json")
    if not run_path.is_file() and "restore_from_drive" in globals():
        restore_from_drive(f"{run_name}_run.json", str(run_path))
    if not run_path.is_file():
        return False
    meta = json.loads(run_path.read_text(encoding="utf-8"))
    expected = {
        "max_steps": max_steps,
        "base_setup_name": base_setup_name.upper(),
        "lora_rank": lora_rank,
        "coord_scale": coord_scale,
        "train_fraction": float(train_fraction),
    }
    if any(meta.get(key) != value for key, value in expected.items()):
        print(f"Evidence {run_name} không khớp recipe E4 hiện tại; sẽ train lại.")
        return False
    return _load_valid_evidence(run_name, "test", meta_info=meta, coord_scale=coord_scale) is not None


def run_required_acceptance_suite(max_steps: int = 300, e4_variable: str = "rank", run_e5: bool = False):
    """Chạy E1–E3, chọn E4 trên validation, chấm test và gọi checker chính thức."""
    if max_steps < 1:
        raise ValueError("max_steps phải lớn hơn 0")
    if e4_variable not in ("rank", "data_size", "coord_scale"):
        raise ValueError("e4_variable phải là rank, data_size hoặc coord_scale")
    _expected_dataset_version()
    if _load_valid_evidence("E0", "test", coord_scale="norm1000") is None:
        raise FileNotFoundError("Thiếu report/evidence/e0.json; hoàn thành E0 trước.")

    for name in ("E1", "E2", "E3"):
        meta = _load_or_train_adapter(name, max_steps)
        if _load_valid_evidence(
            name, "val", meta_info=meta, coord_scale=meta.get("coord_scale", "norm1000")
        ) is None:
            run_eval(
                name, meta_info=meta, split="val", batch_size=1,
                coord_scale=meta.get("coord_scale", "norm1000"),
            )

    best_setup = _best_setup_from_validation()
    print(f"Setup nền E4 được chọn theo validation: {best_setup}")
    for name in ("E1", "E2", "E3"):
        meta_path = Path(f"work/run_{name.lower()}.json")
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if _load_valid_evidence(
            name, "test", meta_info=meta, coord_scale=meta.get("coord_scale", "norm1000")
        ) is None:
            run_eval(
                name, meta_info=meta, split="test", batch_size=1,
                coord_scale=meta.get("coord_scale", "norm1000"),
            )

    if e4_variable == "rank":
        for rank in (4, 64):
            run_name = f"E4_rank_{rank}"
            if not _e4_result_reusable(run_name, max_steps, best_setup, rank, "norm1000", 1.0):
                run_e4_ablation_rank(rank, base_setup_name=best_setup, max_steps=max_steps)
    elif e4_variable == "data_size":
        for fraction in (0.1, 1.0):
            percent = int(round(fraction * 100))
            run_name = f"E4_data_size_{percent}"
            if not _e4_result_reusable(run_name, max_steps, best_setup, 16, "norm1000", fraction):
                run_e4_ablation_data_size(fraction, base_setup_name=best_setup, max_steps=max_steps)
    else:
        run_name = "E4_coord_scale_pixel"
        if not _e4_result_reusable(run_name, max_steps, best_setup, 16, "pixel", 1.0):
            run_e4_ablation_pixel(base_setup_name=best_setup, max_steps=max_steps)

    if e4_variable == "rank":
        e4_names = ["E4_rank_4", "E4_rank_64"]
        e4_values = [4, 64]
    elif e4_variable == "data_size":
        e4_names = ["E4_data_size_10", "E4_data_size_100"]
        e4_values = [10, 100]
    else:
        e4_names = ["E4_coord_scale_pixel"]
        e4_values = ["norm1000", "pixel"]
    e1_meta = json.loads(Path("work/run_e1.json").read_text(encoding="utf-8"))
    e2_meta = json.loads(Path("work/run_e2.json").read_text(encoding="utf-8"))
    run_context = {
        "dataset_version": _expected_dataset_version(),
        "max_steps": max_steps,
        "e4_variable": e4_variable,
        "e4_values": e4_values,
        "e4_evidence_names": e4_names,
        "e4_baseline_setup": best_setup,
        "e2_minus_e1_trainable_params": (
            e2_meta.get("trainable_params", 0) - e1_meta.get("trainable_params", 0)
        ),
    }
    context_path = Path("work/acceptance_run.json")
    context_path.parent.mkdir(parents=True, exist_ok=True)
    context_path.write_text(json.dumps(run_context, indent=2), encoding="utf-8")
    if "backup_to_drive" in globals():
        backup_to_drive(str(context_path), "acceptance_run.json")
    if run_e5 and _load_valid_evidence("E5", "test") is None:
        train_and_eval_e5_grpo(best_adapter_name=best_setup)
    best_prediction = Path("work") / f"{best_setup.lower()}.jsonl"
    if best_prediction.is_file():
        save_prediction_examples(best_setup, split="test", count=6)
    else:
        print(f"Chưa có {best_prediction}; chưa thể tạo ảnh minh hoạ rubric.")
    return run_lab_acceptance()


def _format_evidence_table(evidence_paths):
    rows = [
        "| Mã | Split | Params trainable | VRAM GB | Phút | Valid JSON | F1 | CI95 | FP/ảnh | F1 pothole | F1 crack | F1 manhole |",
        "|---|---|---:|---:|---:|---:|---:|---|---:|---:|---:|---:|",
    ]
    for path in evidence_paths:
        if not path.is_file():
            continue
        data = json.loads(path.read_text(encoding="utf-8"))
        micro = data.get("micro", {})
        per_class = data.get("per_class", {})
        ci = data.get("f1_ci95", [0.0, 0.0])
        meta = data.get("meta", {})
        fmt = lambda value: f"{value:.3f}" if isinstance(value, (int, float)) else ""
        rows.append(
            f"| {data.get('name', path.stem)} | {data.get('split', '')} "
            f"| {meta.get('trainable_params', '')} | {fmt(meta.get('peak_vram_gb'))} "
            f"| {fmt(meta.get('train_minutes'))} | {fmt(data.get('parse_valid_rate'))} "
            f"| {fmt(micro.get('f1'))} | [{fmt(ci[0])}, {fmt(ci[1])}] "
            f"| {fmt(data.get('fp_per_image'))} | {fmt(per_class.get('pothole', {}).get('f1'))} "
            f"| {fmt(per_class.get('crack', {}).get('f1'))} | {fmt(per_class.get('manhole', {}).get('f1'))} |"
        )
    return "\n".join(rows)


def save_prediction_examples(setup_name: str, split: str = "test", count: int = 6, seed: int = 42):
    """Vẽ GT/pred, đánh dấu TP/FP/FN, lưu từng ảnh và contact sheet cho REPORT.md."""
    import random
    import re
    import matplotlib.pyplot as plt
    from PIL import Image, ImageDraw
    from scripts.score import iou_xyxy, parse_output

    setup_name = setup_name.upper()
    if split not in ("test", "val") or count < 1:
        raise ValueError("split phải là test/val và count phải lớn hơn 0")
    evidence_name = f"{setup_name.lower()}.json" if split == "test" else f"debug_{setup_name.lower()}_val.json"
    evidence_path = Path("report/evidence") / evidence_name
    prediction_name = f"{setup_name.lower()}.jsonl" if split == "test" else f"val_{setup_name.lower()}.jsonl"
    prediction_path = Path("work") / prediction_name
    if "restore_from_drive" in globals():
        if not evidence_path.is_file():
            restore_from_drive(f"{setup_name}_{split}_evidence.json", str(evidence_path))
        if not prediction_path.is_file():
            restore_from_drive(
                f"{setup_name}_{split}_predictions.jsonl", str(prediction_path)
            )
    if not evidence_path.is_file() or not prediction_path.is_file():
        raise FileNotFoundError(f"Thiếu prediction/evidence cho {setup_name} split={split}")
    coord = json.loads(evidence_path.read_text(encoding="utf-8")).get("coord", "norm1000")
    split_path = Path(f"data/splits/{split}.jsonl")
    with split_path.open(encoding="utf-8") as source:
        entries = [json.loads(line) for line in source if line.strip()]
    with prediction_path.open(encoding="utf-8") as source:
        predictions = {row["image_id"]: row["raw"] for row in (json.loads(line) for line in source if line.strip())}

    def ground_truth(row):
        boxes = []
        for obj in row.get("objects", []):
            xc, yc, width, height = (float(value) for value in obj["yolo"])
            boxes.append({"label": obj["label"], "bbox": (
                (xc - width / 2) * 1000, (yc - height / 2) * 1000,
                (xc + width / 2) * 1000, (yc + height / 2) * 1000)})
        return boxes

    def normalize_prediction(pred, row):
        if coord == "norm1000":
            return pred
        width, height = float(row["width"]), float(row["height"])
        return [{"label": item["label"], "bbox": (
            item["bbox"][0] / width * 1000, item["bbox"][1] / height * 1000,
            item["bbox"][2] / width * 1000, item["bbox"][3] / height * 1000)} for item in pred]

    candidates = []
    for row in entries:
        raw = predictions.get(row["image_id"])
        parsed = parse_output(raw) if raw is not None else None
        gt, pred = ground_truth(row), normalize_prediction(parsed or [], row)
        pairs = []
        for gi, g in enumerate(gt):
            for pi, item in enumerate(pred):
                if g["label"] == item["label"]:
                    overlap = iou_xyxy(g["bbox"], item["bbox"])
                    if overlap >= 0.5:
                        pairs.append((overlap, gi, pi))
        pairs.sort(key=lambda pair: (-pair[0], pair[1], pair[2]))
        used_gt, used_pred = set(), set()
        for _, gi, pi in pairs:
            if gi not in used_gt and pi not in used_pred:
                used_gt.add(gi)
                used_pred.add(pi)
        errors = len(gt) - len(used_gt) + len(pred) - len(used_pred)
        if parsed is None:
            errors += 1
        candidates.append({"row": row, "raw": raw, "parsed": parsed, "gt": gt, "pred": pred,
                           "used_gt": used_gt, "used_pred": used_pred, "errors": errors})

    rng = random.Random(seed)
    rng.shuffle(candidates)
    candidates.sort(key=lambda item: item["errors"], reverse=True)
    selected = candidates[:min(count, len(candidates))]
    output_dir = Path("report/figures")
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    panels = []
    for index, item in enumerate(selected, start=1):
        row = item["row"]
        image_path = Path("data") / row["image_path"]
        with Image.open(image_path) as source_image:
            image = source_image.convert("RGB")
        draw = ImageDraw.Draw(image)
        width, height = image.size
        for gi, gt in enumerate(item["gt"]):
            x1, y1, x2, y2 = gt["bbox"]
            box = (round(x1 * width / 1000), round(y1 * height / 1000),
                   round(x2 * width / 1000), round(y2 * height / 1000))
            color = "lime" if gi in item["used_gt"] else "orange"
            draw.rectangle(box, outline=color, width=3)
            draw.text((box[0], max(0, box[1] - 14)), f"GT {gt['label']} {'TP' if gi in item['used_gt'] else 'FN'}", fill=color)
        for pi, pred in enumerate(item["pred"]):
            x1, y1, x2, y2 = pred["bbox"]
            box = (round(x1 * width / 1000), round(y1 * height / 1000),
                   round(x2 * width / 1000), round(y2 * height / 1000))
            color = "cyan" if pi in item["used_pred"] else "red"
            draw.rectangle(box, outline=color, width=3)
            draw.text((box[0], min(height - 14, box[1] + 2)), f"PRED {pred['label']} {'TP' if pi in item['used_pred'] else 'FP'}", fill=color)
        if item["parsed"] is None:
            draw.text((5, 5), "INVALID JSON", fill="red")
        safe_id = re.sub(r"[^A-Za-z0-9._-]+", "_", str(row["image_id"]))
        output_path = output_dir / f"{setup_name.lower()}_{split}_{index}_{safe_id}.png"
        image.save(output_path)
        saved.append(output_path)
        panels.append((image.copy(), f"{row['image_id']} | FP+FN={item['errors']}"))

    columns = min(3, len(panels))
    rows_count = (len(panels) + columns - 1) // columns
    figure, axes = plt.subplots(rows_count, columns, figsize=(6 * columns, 4.5 * rows_count), squeeze=False)
    for axis, (image, title) in zip(axes.flat, panels):
        axis.imshow(image)
        axis.set_title(title)
        axis.axis("off")
    for axis in axes.flat[len(panels):]:
        axis.axis("off")
    contact_path = output_dir / f"{setup_name.lower()}_{split}_contact_sheet.png"
    figure.tight_layout()
    figure.savefig(contact_path, dpi=140)
    plt.show()
    plt.close(figure)
    saved.append(contact_path)
    if "backup_to_drive" in globals():
        for path in saved:
            backup_to_drive(str(path), f"figure_{path.name}")
    print(f"Đã lưu {len(panels)} ảnh có chú thích và contact sheet tại {output_dir}.")
    return saved


def _e5_diagnostics():
    evidence_path = Path("report/evidence/e5.json")
    prediction_path = Path("work/e5.jsonl")
    split_path = Path("data/splits/test.jsonl")
    if not (evidence_path.is_file() and prediction_path.is_file() and split_path.is_file()):
        return ["E5 reward-hacking audit: chưa có đủ evidence/prediction/test split."]
    from scripts.score import parse_output
    evidence = json.loads(evidence_path.read_text(encoding="utf-8"))
    with split_path.open(encoding="utf-8") as source:
        entries = [json.loads(line) for line in source if line.strip()]
    with prediction_path.open(encoding="utf-8") as source:
        predictions = {row["image_id"]: row["raw"] for row in (json.loads(line) for line in source if line.strip())}
    false_empty = huge_box = invalid = 0
    nonempty_gt = 0
    for entry in entries:
        has_gt = bool(entry.get("objects"))
        nonempty_gt += int(has_gt)
        raw = predictions.get(entry["image_id"])
        parsed = parse_output(raw) if raw is not None else None
        if parsed is None:
            invalid += 1
            continue
        if has_gt and not parsed:
            false_empty += 1
        if evidence.get("coord") == "pixel":
            image_area = max(float(entry["width"]) * float(entry["height"]), 1.0)
            huge_box |= any((b["bbox"][2] - b["bbox"][0]) * (b["bbox"][3] - b["bbox"][1]) / image_area >= 0.8 for b in parsed)
        else:
            huge_box |= any((b["bbox"][2] - b["bbox"][0]) * (b["bbox"][3] - b["bbox"][1]) / 1_000_000 >= 0.8 for b in parsed)
    false_empty_rate = false_empty / nonempty_gt if nonempty_gt else 0.0
    lines = [f"E5 test output: invalid={invalid}/{len(entries)}, false-empty trên ảnh có GT={false_empty}/{nonempty_gt} ({false_empty_rate:.3f}), "
             f"có box chiếm >=80% ảnh={'có' if huge_box else 'không'}."]
    log_path = Path("work/e5_grpo_log.json")
    if log_path.is_file():
        log = json.loads(log_path.read_text(encoding="utf-8"))
        reward_rows = [{k: v for k, v in row.items() if "reward" in k.lower()} for row in log if isinstance(row, dict)]
        reward_rows = [row for row in reward_rows if row]
        lines.append(f"E5 trainer log: {len(log)} entries; reward fields cuối={reward_rows[-1] if reward_rows else 'chưa có'}.")
    sft = []
    for name in ("e1", "e2", "e3"):
        path = Path(f"report/evidence/{name}.json")
        if path.is_file():
            sft.append(json.loads(path.read_text(encoding="utf-8")))
    if sft:
        best = max(sft, key=lambda row: row.get("micro", {}).get("f1", 0.0))
        f1_e5 = evidence.get("micro", {}).get("f1", 0.0)
        f1_sft = best.get("micro", {}).get("f1", 0.0)
        ci_e5, ci_sft = evidence.get("f1_ci95", [0.0, 0.0]), best.get("f1_ci95", [0.0, 0.0])
        overlap = max(ci_e5[0], ci_sft[0]) <= min(ci_e5[1], ci_sft[1])
        lines.append(f"E5 vs {best.get('name')}: ΔF1={f1_e5 - f1_sft:+.3f}; CI95 chồng lấp={'có' if overlap else 'không'} (mô tả, không phải kiểm định ý nghĩa thống kê).")
    return lines


def save_training_loss_plot():
    """Vẽ loss E1/E2/E3 từ log thật để dùng trong báo cáo."""
    logs = {}
    for name in ("E1", "E2", "E3"):
        path = Path(f"work/{name.lower()}_loss.json")
        if not path.is_file() and "restore_from_drive" in globals():
            restore_from_drive(f"{name}_loss.json", str(path))
        if path.is_file():
            rows = json.loads(path.read_text(encoding="utf-8"))
            if rows:
                logs[name] = rows
    if not logs:
        return None
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("WARN: thiếu matplotlib; không tạo được loss plot.")
        return None
    output_dir = Path("report/figures")
    output_dir.mkdir(parents=True, exist_ok=True)
    figure, axis = plt.subplots(figsize=(9, 5))
    for name, rows in logs.items():
        axis.plot([row["step"] for row in rows], [row["loss"] for row in rows], label=name)
    axis.set(title="SFT training loss", xlabel="Optimizer step", ylabel="Loss")
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()
    output = output_dir / "training_loss.png"
    figure.savefig(output, dpi=150)
    plt.show()
    plt.close(figure)
    print(f"Đã lưu loss plot: {output}")
    if "backup_to_drive" in globals():
        backup_to_drive(str(output), "figure_training_loss.png")
    return output


def save_e4_ablation_plot():
    """Vẽ F1 và CI95 của baseline với các giá trị E4 đã chọn."""
    context_path = Path("work/acceptance_run.json")
    if not context_path.is_file() and "restore_from_drive" in globals():
        restore_from_drive("acceptance_run.json", str(context_path))
    if not context_path.is_file():
        return None
    context = json.loads(context_path.read_text(encoding="utf-8"))
    variable = context.get("e4_variable")
    baseline = context.get("e4_baseline_setup", "E1").lower()
    candidates = [(context.get("e4_baseline_setup", "E1"), Path(f"report/evidence/{baseline}.json"))]
    for name in context.get("e4_evidence_names", []):
        candidates.append((name.replace("E4_", "").replace("_", "="), Path("report/evidence") / f"{name.lower()}.json"))
    records = []
    for label, path in candidates:
        if path.is_file():
            data = json.loads(path.read_text(encoding="utf-8"))
            f1 = data.get("micro", {}).get("f1")
            ci = data.get("f1_ci95")
            if isinstance(f1, (int, float)) and isinstance(ci, list) and len(ci) == 2:
                records.append((label, float(f1), float(ci[0]), float(ci[1])))
    if len(records) < 2:
        return None
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("WARN: thiếu matplotlib; không tạo được E4 CI plot.")
        return None
    output_dir = Path("report/figures")
    output_dir.mkdir(parents=True, exist_ok=True)
    figure, axis = plt.subplots(figsize=(8, 5))
    x = list(range(len(records)))
    scores = [row[1] for row in records]
    lower = [max(0.0, row[1] - row[2]) for row in records]
    upper = [max(0.0, row[3] - row[1]) for row in records]
    axis.errorbar(x, scores, yerr=[lower, upper], fmt="o", capsize=5, linewidth=2)
    axis.set_xticks(x, [row[0] for row in records], rotation=15)
    axis.set_ylim(0, 1)
    axis.set(title=f"E4 {variable}: F1 with 95% bootstrap CI", ylabel="Micro F1")
    axis.grid(axis="y", alpha=0.25)
    figure.tight_layout()
    output = output_dir / "e4_f1_ci95.png"
    figure.savefig(output, dpi=150)
    plt.show()
    plt.close(figure)
    print(f"Đã lưu E4 F1/CI95 plot: {output}")
    if "backup_to_drive" in globals():
        backup_to_drive(str(output), "figure_e4_f1_ci95.png")
    return output


def update_report_metric_table():
    """Cập nhật riêng bảng số liệu trong REPORT.md từ evidence thật; không tự trả lời câu hỏi."""
    report_path = Path("REPORT.md")
    evidence_dir = Path("report/evidence")
    required = [evidence_dir / f"{name}.json" for name in ("e0", "e1", "e2", "e3")]
    if not report_path.is_file() or not all(path.is_file() for path in required):
        return False
    context_path = Path("work/acceptance_run.json")
    context = json.loads(context_path.read_text(encoding="utf-8")) if context_path.is_file() else {}
    e4_names = context.get("e4_evidence_names", [])
    e4_paths = [evidence_dir / f"{name.lower()}.json" for name in e4_names]
    e4_paths = [path for path in e4_paths if path.is_file()]
    if not e4_paths:
        e4_paths = sorted(evidence_dir.glob("e4*.json"))
    if not e4_paths:
        return False

    def fmt(value):
        return f"{value:.3f}" if isinstance(value, (int, float)) else ""

    def row_for(path, setup):
        data = json.loads(path.read_text(encoding="utf-8"))
        meta, micro = data.get("meta", {}), data.get("micro", {})
        ci = data.get("f1_ci95", [None, None])
        classes = data.get("per_class", {})
        params = meta.get("trainable_params", 0 if data.get("name", "").upper() == "E0" else "")
        return (
            f"| {data.get('name', path.stem)} | {setup} | {params} | "
            f"{fmt(meta.get('peak_vram_gb'))} | {fmt(meta.get('train_minutes'))} | "
            f"{fmt(micro.get('f1'))} | [{fmt(ci[0])}, {fmt(ci[1])}] | "
            f"{fmt(data.get('fp_per_image'))} | {fmt(classes.get('pothole', {}).get('f1'))} | "
            f"{fmt(classes.get('crack', {}).get('f1'))} | {fmt(classes.get('manhole', {}).get('f1'))} |"
        )

    setups = {"e0": "zero-shot", "e1": "LoRA LLM", "e2": "+ projector", "e3": "+ LoRA ViT"}
    rows = [row_for(path, setups[path.stem]) for path in required]
    for path in e4_paths:
        suffix = path.stem.removeprefix("e4_")
        if suffix.startswith("rank_"):
            label = "rank=" + suffix.removeprefix("rank_")
        elif suffix.startswith("data_size_"):
            label = "data_size=" + suffix.removeprefix("data_size_") + "%"
        elif suffix == "coord_scale_pixel":
            label = "coord_scale=pixel"
        else:
            label = suffix
        rows.append(row_for(path, label))

    text = report_path.read_text(encoding="utf-8")
    lines = text.splitlines()
    start = next((i for i, line in enumerate(lines) if line.startswith("| Mã | Setup |")), None)
    if start is None:
        print("WARN: không tìm thấy bảng tổng hợp trong REPORT.md; không sửa file.")
        return False
    end = start + 2
    while end < len(lines) and lines[end].startswith("|"):
        end += 1
    lines[start:end] = [lines[start], lines[start + 1], *rows]
    text = "\n".join(lines) + "\n"
    version_path = Path("data/splits/meta.json")
    if version_path.is_file():
        version = json.loads(version_path.read_text(encoding="utf-8"))["dataset_version"]
        text = text.replace("`dataset_version`: <điền, lấy từ data/splits/meta.json>", f"`dataset_version`: {version}")
    report_path.write_text(text, encoding="utf-8")
    if "backup_to_drive" in globals():
        backup_to_drive(str(report_path), "REPORT_with_metrics.md")
    print("Đã cập nhật bảng metrics và dataset_version trong REPORT.md; 10 câu hỏi vẫn cần phần phân tích của bạn.")
    return True


def _report_artifact_lines():
    lines = []
    audit_path = Path("report/data_audit.json")
    if audit_path.is_file():
        audit = json.loads(audit_path.read_text(encoding="utf-8"))
        lines.append(f"- Dataset: `{audit.get('dataset_version')}`; session disjoint: {audit.get('session_disjoint')}.")
        for split, stats in audit.get("splits", {}).items():
            lines.append(
                f"- {split}: {stats['images']} ảnh, {stats['boxes']} boxes, "
                f"{stats['sessions']} sessions; boxes theo lớp: {stats['boxes_by_class']}."
            )
    else:
        lines.append("- Split audit chưa được tạo.")
    for label, path in (
        ("E0 raw examples", Path("report/evidence/e0_raw_examples.json")),
        ("Coordinate round-trip", Path("report/coordinate_roundtrip.json")),
        ("Coordinate-scale mismatch diagnostic", Path("report/coordinate_scale_diagnostic.json")),
        ("Training example token/mask check", Path("report/training_example_check.json")),
    ):
        lines.append(f"- {label}: `{path}` ({'có' if path.is_file() else 'chưa có'}).")
    for name in ("e1", "e2", "e3"):
        run_path = Path(f"work/run_{name}.json")
        if run_path.is_file():
            run = json.loads(run_path.read_text(encoding="utf-8"))
            architecture = run.get("architecture_params", {})
            setup = run.get("setup_details", {})
            if architecture:
                lines.append(
                    f"- {name.upper()} architecture params: LLM+lm_head={architecture.get('language_model_plus_lm_head')}, "
                    f"visual={architecture.get('visual_total_including_projector')}, "
                    f"projector={architecture.get('projector_merger_and_deepstack')}; "
                    f"LoRA rank={setup.get('lora_rank')}, target modules={setup.get('lora_target_module_count')}, "
                    f"projector dtype={setup.get('projector_trainable_dtype')}."
                )
        path = Path(f"work/{name}_loss.json")
        if path.is_file():
            rows = json.loads(path.read_text(encoding="utf-8"))
            if rows:
                lines.append(
                    f"- {name.upper()} loss: {rows[0]['loss']:.4f} → {rows[-1]['loss']:.4f} "
                    f"({len(rows)} steps); log `{path}`."
                )
    context_path = Path("work/acceptance_run.json")
    if context_path.is_file():
        context = json.loads(context_path.read_text(encoding="utf-8"))
        lines.append(
            f"- E4: biến `{context['e4_variable']}`, giá trị {context['e4_values']}, "
            f"baseline `{context['e4_baseline_setup']}`; E2−E1 trainable params="
            f"{context.get('e2_minus_e1_trainable_params')}."
        )
    figure_files = sorted(Path("report/figures").glob("*.png")) if Path("report/figures").exists() else []
    if figure_files:
        lines.append("- Figures: " + ", ".join(f"`{path}`" for path in figure_files) + ".")
    return "\n".join(lines)


def create_report_handoff_bundle():
    """Gói evidence cần phân tích; loại trừ dataset, credentials và model weights."""
    from zipfile import ZIP_DEFLATED, ZipFile

    output = Path("report/day10_report_handoff.zip")
    output.parent.mkdir(parents=True, exist_ok=True)
    candidates = [
        Path("README.md"), Path("GUIDE.md"), Path("RUBRIC.md"),
        Path("REPORT.md"), Path("lab_day10.ipynb"),
        Path("report/acceptance_summary.md"),
        Path("report/data_audit.json"), Path("report/coordinate_roundtrip.json"),
        Path("report/coordinate_scale_diagnostic.json"), Path("report/training_example_check.json"),
    ]
    for directory, pattern in (
        (Path("report/evidence"), "*.json"),
        (Path("report/diagnostics"), "*.json"),
        (Path("report/figures"), "*.png"),
    ):
        if directory.is_dir():
            candidates.extend(sorted(directory.glob(pattern)))
    work = Path("work")
    if work.is_dir():
        for pattern in ("e*.jsonl", "val_e*.jsonl", "run_*.json", "*_loss.json", "e5_grpo_log.json", "acceptance_run.json"):
            candidates.extend(sorted(work.glob(pattern)))
    files = sorted({path for path in candidates if path.is_file() and path != output})
    version_path = Path("data/splits/meta.json")
    manifest = {
        "dataset_version": json.loads(version_path.read_text(encoding="utf-8")).get("dataset_version") if version_path.is_file() else None,
        "included_files": [path.as_posix() for path in files],
        "excluded": ["data/", "model checkpoints/adapters", "credentials/secrets"],
    }
    with ZipFile(output, "w", compression=ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, arcname=path.as_posix())
        archive.writestr("report/handoff_manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
    print(f"REPORT handoff: {output} ({len(files)} files, {output.stat().st_size / (1024**2):.2f} MiB)")
    if "backup_to_drive" in globals():
        backup_to_drive(str(output), "day10_report_handoff.zip")
    return output


def run_lab_acceptance(suite_error: str | None = None):
    """In ra kết quả thật đang có, chạy checker, lưu summary và các gate số của RUBRIC."""
    evidence_dir = Path("report/evidence")
    paths = [evidence_dir / f"{name}.json" for name in ("e0", "e1", "e2", "e3")]
    paths += sorted(evidence_dir.glob("e4*.json"))
    paths += [evidence_dir / "e5.json"]
    report_update = update_report_metric_table()
    plot_paths = []
    for plot_function in (save_training_loss_plot, save_e4_ablation_plot):
        try:
            plot_path = plot_function()
            if plot_path is not None:
                plot_paths.append(str(plot_path))
        except Exception as exc:
            print(f"WARN: không tạo được một biểu đồ báo cáo: {type(exc).__name__}: {exc}")
    table = _format_evidence_table(paths)
    print(table)
    for diagnostic in _e5_diagnostics():
        print(diagnostic)
    figure_dir = Path("report/figures")
    prediction_figures = (
        [path for path in figure_dir.glob("*_test_*.png")
         if re.search(r"_test_\d+_", path.name)]
        if figure_dir.exists() else []
    )
    figure_count = len(prediction_figures)
    figure_gate = figure_count >= 6
    print(f"[{'PASS' if figure_gate else 'BLOCKED'}] RUBRIC annotated prediction images >= 6 ({figure_count})")

    results = {}
    for name in ("e0", "e1", "e2", "e3"):
        path = evidence_dir / f"{name}.json"
        results[name] = json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None
    gates = {}
    e0 = results["e0"]
    e1 = results["e1"]
    gates["E1 parse_valid_rate >= 0.95"] = (
        None if e1 is None else e1.get("parse_valid_rate", 0) >= 0.95
    )
    gates["E1 F1 >= E0 F1 + 0.10"] = (
        None if e0 is None or e1 is None else
        e1.get("micro", {}).get("f1", 0) >= e0.get("micro", {}).get("f1", 0) + 0.10
    )
    gates["at least 6 annotated prediction images"] = figure_gate
    for label, passed in gates.items():
        status = "BLOCKED" if passed is None else ("PASS" if passed else "FAIL")
        print(f"[{status}] RUBRIC {label}")

    checker = subprocess.run(
        [sys.executable, "scripts/check_submission.py"],
        capture_output=True,
        text=True,
        check=False,
    )
    print(checker.stdout, end="")
    if checker.stderr:
        print(checker.stderr, file=sys.stderr)
    summary_path = Path("report/acceptance_summary.md")
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    gate_lines = [f"- {'BLOCKED' if value is None else ('PASS' if value else 'FAIL')}: {label}"
                  for label, value in gates.items()]
    diagnostic_lines = [f"- {line}" for line in _e5_diagnostics()]
    error_section = (f"## Training suite error\n\n{suite_error}\n\n" if suite_error else "")
    summary_path.write_text(
        "# Day 10 acceptance status\n\n" + error_section + table + "\n\n" +
        "## Inputs and figures for REPORT.md\n\n" + _report_artifact_lines() + "\n\n" +
        f"REPORT.md numeric table updated: {report_update}\n\n" +
        "## Rubric numeric gates\n\n" + "\n".join(gate_lines) + "\n\n" +
        "## E5 diagnostic\n\n" + "\n".join(diagnostic_lines) + "\n\n" +
        "## Download\n\nAfter a full run, download `/content/K4-Track-4-Day10-VisionLanguageModel/report/day10_report_handoff.zip` from Colab Files or `MyDrive/AI20K_Lab_Day10_Checkpoints/day10_report_handoff.zip` from Drive; extract at repo root. The ZIP contains lab instructions and report artifacts, and excludes dataset, credentials and model weights.\n\n" +
        "## Official checker\n\n```text\n" + checker.stdout + "```\n",
        encoding="utf-8",
    )
    bundle_path = create_report_handoff_bundle()
    if "backup_to_drive" in globals():
        backup_to_drive(str(summary_path), "acceptance_summary.md")
    print(f"Báo cáo nghiệm thu lưu tại {summary_path}; tải handoff tại {bundle_path}.")
    return {"checker_exit_code": checker.returncode, "gates": gates, "summary_path": str(summary_path), "handoff_bundle": str(bundle_path)}


# Run all now performs E1–E4 and writes real acceptance artifacts.
ACCEPTANCE_MAX_STEPS = 300
ACCEPTANCE_E4_VARIABLE = "rank"
ACCEPTANCE_RUN_E5 = False
try:
    acceptance_status = run_required_acceptance_suite(
        max_steps=ACCEPTANCE_MAX_STEPS,
        e4_variable=ACCEPTANCE_E4_VARIABLE,
        run_e5=ACCEPTANCE_RUN_E5,
    )
except Exception as exc:
    suite_error = f"{type(exc).__name__}: {exc}"
    print(f"E1–E4 acceptance run stopped: {suite_error}")
    acceptance_status = run_lab_acceptance(suite_error=suite_error)
    raise RuntimeError(
        "E1–E4 chưa hoàn tất; trạng thái checker đã lưu ở report/acceptance_summary.md"
    ) from exc
